# 🌿 Deteksi & Klasifikasi Penyakit Daun Teh (Optimized Classification): YOLO26n + Coordinate Attention (CA)
Notebook terpadu (**Classification-Optimized Pipeline**) yang mengimplementasikan **3 Strategi Peningkatan Akurasi Klasifikasi Kondisi Daun**:
1. **Strategi 1 — Targeted Class-Balanced Oversampling (`train` only)**: Menyeimbangkan kelas minoritas (`Tea algal leaf spot`, `Brown Blight`, `Helopeltis`, `Red Spider`) menjadi minimal 650 sampel di split `train` tanpa menyentuh `val`/`test` (0% *data leakage*, tanpa menghapus data asli).
2. **Strategi 2 — Color-Preserving & Multi-Orientation Classification Tuning**: Mengunci kemurnian warna bercak penyakit (`hsv_h=0.004`), ketahanan bayangan (`hsv_s=0.35, hsv_v=0.35`), orientasi daun penuh (`degrees=30, flipud=0.5, fliplr=0.5`), bobot klasifikasi tinggi (`cls=1.25`), dan `close_mosaic=15`.
3. **Strategi 3 — Pure Model-Driven 4-View TTA + Adaptive Shadow CLAHE**: Menghapus aturan warna kaku yang membajak prediksi model, diganti dengan normalisasi bayangan lembut (LAB-CLAHE) dan konsensus **4-View TTA** (*Original, Flip-H, Flip-V, Rotate-90°*).

### 1. Inisialisasi Environment, Google Drive & Ekstraksi Dataset

In [ ]:
import os
import sys
import glob
import random
import shutil
import zipfile
from pathlib import Path

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install library yang dibutuhkan
!pip install -q ultralytics opencv-python-headless matplotlib pandas

import numpy as np
import torch
print(f"[INFO] PyTorch: {torch.__version__} | CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

# 3. Cari dan Ekstrak Dataset tea_yolo.zip dari Google Drive
DATA_DIR = "/content/tea_yolo"
DATA_YAML = "/content/tea_yolo/data.yaml"

if not os.path.exists(DATA_YAML):
    zip_candidates = (
        glob.glob('/content/drive/MyDrive/SKRIPSI/tea_yolo.zip') +
        glob.glob('/content/drive/MyDrive/**/tea_yolo.zip', recursive=True)
    )
    if not zip_candidates:
        raise FileNotFoundError("Berkas 'tea_yolo.zip' tidak ditemukan di Google Drive!")

    zip_path = zip_candidates[0]
    print(f"[INFO] Mengekstrak dataset dari: {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')
    print(f"[OK] Dataset berhasil diekstrak ke: {DATA_DIR}")
else:
    print(f"[OK] Dataset sudah tersedia di: {DATA_DIR}")

# 4. Sesuaikan path pada data.yaml agar menunjuk ke /content/tea_yolo
with open(DATA_YAML, "r", encoding="utf-8") as f:
    y_txt = f.read()
y_txt = y_txt.replace("D:/SKRIPSI/tea_yolo", "/content/tea_yolo").replace("D:\\SKRIPSI\\tea_yolo", "/content/tea_yolo")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(y_txt)
print("[OK] data.yaml siap digunakan.")

# 5. Siapkan bobot pretrained dasar yolo26n.pt
if not os.path.exists('/content/yolo26n.pt'):
    pt_candidates = (
        glob.glob('/content/drive/MyDrive/SKRIPSI/**/yolo26n.pt', recursive=True) +
        glob.glob('/content/drive/MyDrive/**/yolo26n.pt', recursive=True)
    )
    if pt_candidates:
        shutil.copy(pt_candidates[0], '/content/yolo26n.pt')
        print(f"[OK] yolo26n.pt disalin dari: {pt_candidates[0]}")
    else:
        from ultralytics import YOLO
        _tmp = YOLO('yolo26n.pt')
        print("[OK] yolo26n.pt berhasil diunduh via Ultralytics.")


### 2. Definisi Arsitektur YOLO26n + Coordinate Attention (CA) & Strategi 1 (Class Balancing `train`)

In [ ]:
import copy
import types
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.modules as ult_modules
import ultralytics.nn.tasks as ult_tasks

CLASS_NAMES = [
    "Tea algal leaf spot", "Brown Blight", "Gray Blight",
    "Helopeltis", "Red Spider", "Green Mirid Bug", "Healty Leaf"
]
CLASS_COLORS = [
    (0, 140, 255),   # 0: Tea algal leaf spot - Orange
    (0, 0, 255),     # 1: Brown Blight        - Red
    (0, 215, 255),   # 2: Gray Blight         - Gold/Yellow
    (255, 0, 255),   # 3: Helopeltis          - Magenta
    (50, 50, 255),   # 4: Red Spider          - Coral Red
    (0, 255, 0),     # 5: Green Mirid Bug     - Green
    (255, 255, 0),   # 6: Healty Leaf         - Cyan
]

class CoordAtt(nn.Module):
    """Coordinate Attention Block (Hou et al., CVPR 2021)."""
    def __init__(self, inp: int = 128, oup: int = None, reduction: int = 32):
        super().__init__()
        oup = oup or inp
        self.inp = inp
        self.oup = oup
        self.reduction = reduction
        self.act = nn.Hardswish()
        self._init_layers(inp, oup)

    def _init_layers(self, inp: int, oup: int):
        mip = max(8, inp // self.reduction)
        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0, bias=True)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        B, C, H, W = x.shape
        if self.conv1.in_channels != C:
            self._init_layers(C, C)
            self.to(device=x.device, dtype=x.dtype)
        x_h = torch.mean(x, dim=3, keepdim=True)
        x_w = torch.mean(x, dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h_enc, x_w_enc = torch.split(y, [H, W], dim=2)
        x_w_enc = x_w_enc.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h_enc).sigmoid()
        a_w = self.conv_w(x_w_enc).sigmoid()
        return identity * a_h * a_w

class CoordAttYOLO(CoordAtt):
    """Wrapper CoordAtt untuk parser YAML Ultralytics."""
    def __init__(self, c1: int = 128, reduction: int = 32, *args, **kwargs):
        if isinstance(c1, (list, tuple)):
            c1 = c1[0]
        if "inp" in kwargs:
            c1 = kwargs["inp"]
        if "reduction" in kwargs:
            reduction = kwargs["reduction"]
        super().__init__(inp=int(c1), oup=int(c1), reduction=int(reduction))

def register_ca_everywhere():
    setattr(nn, "CoordAtt", CoordAtt)
    setattr(nn, "CoordAttYOLO", CoordAttYOLO)
    setattr(ult_modules, "CoordAtt", CoordAtt)
    setattr(ult_modules, "CoordAttYOLO", CoordAttYOLO)
    setattr(ult_tasks, "CoordAtt", CoordAtt)
    setattr(ult_tasks, "CoordAttYOLO", CoordAttYOLO)
    ult_tasks.__dict__["CoordAtt"] = CoordAtt
    ult_tasks.__dict__["CoordAttYOLO"] = CoordAttYOLO
    if hasattr(ult_tasks, "parse_model"):
        ult_tasks.parse_model.__globals__["CoordAtt"] = CoordAtt
        ult_tasks.parse_model.__globals__["CoordAttYOLO"] = CoordAttYOLO
    curr_mod = sys.modules[__name__]
    for pkg in ("phase3_yolo_ca", "phase3_yolo_ca_finetune", "phase3_scripts"):
        if pkg not in sys.modules:
            p_obj = types.ModuleType(pkg)
            p_obj.ca_module = curr_mod
            sys.modules[pkg] = p_obj
        else:
            setattr(sys.modules[pkg], "ca_module", curr_mod)
        sys.modules[f"{pkg}.ca_module"] = curr_mod

register_ca_everywhere()

def balance_minority_train_split(data_dir: str, target_min: int = 650, seed: int = 42):
    """
    STRATEGI 1: Targeted Oversampling HANYA pada split 'train' agar kelas minoritas
    (Tea algal leaf spot, Brown Blight, Helopeltis, Red Spider) seimbang.
    Split 'val' dan 'test' TIDAK disentuh sama sekali (0% data leakage, tanpa menghapus data asli).
    """
    img_tr = os.path.join(data_dir, "images", "train")
    lbl_tr = os.path.join(data_dir, "labels", "train")

    # Hapus file bal_* lama agar idempoten jika cell dijalankan ulang
    for old_f in glob.glob(os.path.join(img_tr, "bal_*")) + glob.glob(os.path.join(lbl_tr, "bal_*")):
        os.remove(old_f)
    for cache_f in glob.glob(os.path.join(data_dir, "labels", "*.cache")):
        os.remove(cache_f)

    cls_files = {c: [] for c in range(7)}
    for lbl_p in sorted(glob.glob(os.path.join(lbl_tr, "*.txt"))):
        bname = os.path.splitext(os.path.basename(lbl_p))[0]
        if bname.startswith(("bal_", "lapangan_", "rf_")):
            continue
        with open(lbl_p, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if not lines:
            continue
        cid = int(lines[0].split()[0])
        if cid in cls_files:
            for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
                ip = os.path.join(img_tr, bname + ext)
                if os.path.exists(ip):
                    cls_files[cid].append((ip, lbl_p))
                    break

    rng = random.Random(seed)
    added = 0
    print("\n[STRATEGI 1] Penyeimbangan Kelas Minoritas pada Split 'train' (Val & Test Murni):")
    for cid in range(7):
        orig_list = cls_files[cid]
        n_orig = len(orig_list)
        if 0 < n_orig < target_min:
            need = target_min - n_orig
            picks = [orig_list[i % n_orig] for i in range(need)]
            rng.shuffle(picks)
            for k, (src_img, src_lbl) in enumerate(picks, 1):
                ext = os.path.splitext(src_img)[1]
                stem = os.path.splitext(os.path.basename(src_img))[0]
                shutil.copy2(src_img, os.path.join(img_tr, f"bal_{cid}_{k:04d}_{stem}{ext}"))
                shutil.copy2(src_lbl, os.path.join(lbl_tr, f"bal_{cid}_{k:04d}_{stem}.txt"))
                added += 1
            print(f"  - Kelas {cid} ({CLASS_NAMES[cid]:<20}): {n_orig:4d} -> {n_orig + need:4d} (+{need} oversampled)")
        else:
            print(f"  - Kelas {cid} ({CLASS_NAMES[cid]:<20}): {n_orig:4d} -> {n_orig:4d} (mayoritas, tetap)")
    print(f"[OK] Total citra train siap latih: {len(glob.glob(os.path.join(img_tr, '*.*')))} (+{added} sampel minoritas)")

balance_minority_train_split(DATA_DIR, target_min=650, seed=42)


### 3. File Konfigurasi `yolo26n_ca.yaml` & Pemetaan Bobot Pretrained (100% Remapping)

In [ ]:
MODEL_YAML = "/content/yolo26n_ca.yaml"
YAML_CONTENT = """nc: 7
end2end: True
reg_max: 1

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]             # 0 - P1/2
  - [-1, 1, Conv, [128, 3, 2]]            # 1 - P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]     # 2 - P2/4
  - [-1, 1, Conv, [256, 3, 2]]            # 3 - P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]     # 4 - P3/8 (out_ch=128)
  - [-1, 1, CoordAttYOLO, [128, 32]]      # 5 - CA-P3 (out_ch=128)

  - [-1, 1, Conv, [512, 3, 2]]            # 6 - P4/16
  - [-1, 2, C3k2, [512, True]]            # 7 - P4/16 (out_ch=128)
  - [-1, 1, CoordAttYOLO, [128, 32]]      # 8 - CA-P4 (out_ch=128)

  - [-1, 1, Conv, [1024, 3, 2]]           # 9 - P5/32
  - [-1, 2, C3k2, [1024, True]]           # 10 - P5/32
  - [-1, 1, SPPF, [1024, 5, 3, True]]     # 11 - P5/32
  - [-1, 2, C2PSA, [1024]]                # 12 - P5/32 (out_ch=256)
  - [-1, 1, CoordAttYOLO, [256, 32]]      # 13 - CA-P5 (out_ch=256)

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 14
  - [[-1, 8], 1, Concat, [1]]                  # 15 cat P4 CA
  - [-1, 2, C3k2, [512, True]]                 # 16

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 17
  - [[-1, 5], 1, Concat, [1]]                  # 18 cat P3 CA
  - [-1, 2, C3k2, [256, True]]                 # 19

  - [-1, 1, Conv, [256, 3, 2]]                 # 20
  - [[-1, 16], 1, Concat, [1]]                 # 21 cat head P4
  - [-1, 2, C3k2, [512, True]]                 # 22

  - [-1, 1, Conv, [512, 3, 2]]                 # 23
  - [[-1, 13], 1, Concat, [1]]                 # 24 cat P5 CA
  - [-1, 1, C3k2, [1024, True, 0.5, True]]     # 25

  - [[19, 22, 25], 1, Detect, [nc]]            # 26 Detect(P3, P4, P5)
"""
with open(MODEL_YAML, "w", encoding="utf-8") as f:
    f.write(YAML_CONTENT)

REMAPPED_PT = "/content/yolo26n_ca_coco_init.pt"
LAYER_MAP = {
    0: 0, 1: 1, 2: 2, 3: 3, 4: 4,
    5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12,
    11: 14, 12: 15, 13: 16, 14: 17, 15: 18, 16: 19,
    17: 20, 18: 21, 19: 22, 20: 23, 21: 24, 22: 25, 23: 26
}

def remap_yolo26n_to_ca(base_pt, yaml_path, out_pt):
    ca_model = YOLO(yaml_path).model
    ca_sd = ca_model.state_dict()
    ckpt = torch.load(base_pt, map_location="cpu", weights_only=False)
    base_sd = (ckpt.get("model") or ckpt.get("ema")).float().state_dict()

    new_sd = copy.deepcopy(ca_sd)
    matched = 0
    for old_k, tensor in base_sd.items():
        parts = old_k.split(".")
        if len(parts) >= 3 and parts[0] == "model" and parts[1].isdigit():
            old_idx = int(parts[1])
            new_idx = LAYER_MAP.get(old_idx)
            if new_idx is not None:
                new_k = ".".join(["model", str(new_idx)] + parts[2:])
                if new_k in new_sd and new_sd[new_k].shape == tensor.shape:
                    new_sd[new_k] = tensor.to(dtype=new_sd[new_k].dtype)
                    matched += 1

    ca_model.load_state_dict(new_sd, strict=True)
    orig_train_args = ckpt.get("train_args")
    if not isinstance(orig_train_args, dict):
        orig_train_args = {}

    save_obj = {
        "epoch": -1,
        "best_fitness": None,
        "model": copy.deepcopy(ca_model).half(),
        "ema": None,
        "updates": None,
        "optimizer": None,
        "train_args": orig_train_args,
        "date": None,
        "version": ckpt.get("version", "8.4.178"),
    }
    torch.save(save_obj, out_pt)
    print(f"[OK] Berhasil memetakan {matched}/{len(base_sd)} bobot tensor ke {out_pt}")

remap_yolo26n_to_ca('/content/yolo26n.pt', MODEL_YAML, REMAPPED_PT)


### 4. Pelatihan Model YOLO26n + CA dengan Strategi 2 (Color-Preserving + Multi-Orientation + `cls=1.25`)

In [ ]:
IS_FINETUNE = False
EPOCHS = 30 if IS_FINETUNE else 50
BATCH = 16
IMGSZ = 640
RUN_NAME = "yolo26n_ca_cls_opt_finetune" if IS_FINETUNE else "yolo26n_ca_cls_opt_50ep"
PROJECT_RUNS = "/content/phase3_cls_opt_runs"

if IS_FINETUNE:
    s1_weights = (
        glob.glob(f"{PROJECT_RUNS}/yolo26n_ca_cls_opt_50ep/weights/best.pt") +
        glob.glob("/content/drive/MyDrive/SKRIPSI/**/yolo26n_ca_cls_opt_50ep/**/best.pt", recursive=True)
    )
    init_weights = s1_weights[0] if s1_weights else REMAPPED_PT
    print(f"[INFO] Mode Fine-Tuning aktif. Memuat bobot dasar: {init_weights}")
else:
    init_weights = REMAPPED_PT
    print(f"[INFO] Mode Training Classification-Optimized (50 Epoch) aktif. Memuat: {init_weights}")

model = YOLO(MODEL_YAML)

# STRATEGI 2: Color-Preserving Augmentation + Orientasi Penuh + Bobot Klasifikasi Tinggi
train_kwargs = dict(
    data=DATA_YAML,
    pretrained=init_weights,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=8,
    device="cuda" if torch.cuda.is_available() else "cpu",
    project=PROJECT_RUNS,
    name=RUN_NAME,
    exist_ok=True,
    patience=15,
    save_period=10,
    seed=42,
    cos_lr=True,
    cls=1.25,            # Prioritaskan ketajaman klasifikasi 7 kondisi daun
    hsv_h=0.004,         # Kunci warna asli bercak (jingga vs cokelat vs abu-abu)
    hsv_s=0.35,          # Variasi saturasi terhadap cahaya matahari
    hsv_v=0.35,          # Ketahanan terhadap bayangan kebun
    degrees=30.0,        # Invariansi kemiringan daun
    flipud=0.5,          # Flip vertikal (daun bebas orientasi atas-bawah)
    fliplr=0.5,          # Flip horizontal
    scale=0.35,
    close_mosaic=15,     # 15 epoch terakhir fokus citra daun utuh
    verbose=True,
    plots=True,
)

if IS_FINETUNE:
    train_kwargs.update(dict(
        optimizer="AdamW",
        lr0=0.0004,
        lrf=0.01,
        box=8.0,
        cls=1.35,
        mosaic=0.0,
        close_mosaic=EPOCHS,
        freeze=4,
    ))

print(f"[INFO] Memulai training {RUN_NAME} ({EPOCHS} Epoch, 640x640)...")
train_results = model.train(**train_kwargs)
BEST_WEIGHTS = os.path.join(PROJECT_RUNS, RUN_NAME, "weights", "best.pt")
print(f"[SUKSES] Pelatihan selesai! Bobot terbaik disimpan di: {BEST_WEIGHTS}")


### 5. Evaluasi Lengkap pada Held-Out Test Set Murni (529 Citra, 640x640)

In [ ]:
import pandas as pd
from IPython.display import display

eval_model = YOLO(BEST_WEIGHTS)
metrics = eval_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device="cuda" if torch.cuda.is_available() else "cpu",
    project=os.path.join(PROJECT_RUNS, RUN_NAME),
    name="test_eval",
    exist_ok=True,
    save_json=True,
    plots=True,
    verbose=False,
)

mp = float(metrics.box.mp)
mr = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map5095 = float(metrics.box.map)
f1 = 2 * mp * mr / (mp + mr + 1e-9)

rows = []
if hasattr(metrics.box, "ap_class_index") and metrics.box.ap_class_index is not None:
    for i, c_idx in enumerate(metrics.box.ap_class_index):
        cname = CLASS_NAMES[int(c_idx)]
        p_c = float(metrics.box.p[i])
        r_c = float(metrics.box.r[i])
        ap50 = float(metrics.box.ap50[i])
        ap95 = float(metrics.box.ap[i]) if hasattr(metrics.box, "ap") else ap50 * 0.95
        f1_c = 2 * p_c * r_c / (p_c + r_c + 1e-9)
        rows.append({
            "Kelas": cname,
            "Precision": f"{p_c:.4f}",
            "Recall": f"{r_c:.4f}",
            "F1-Score": f"{f1_c:.4f}",
            "mAP@0.50": f"{ap50:.4f}",
            "mAP@0.50:0.95": f"{ap95:.4f}",
        })

rows.append({
    "Kelas": "== MEAN / OVERALL ==",
    "Precision": f"{mp:.4f}",
    "Recall": f"{mr:.4f}",
    "F1-Score": f"{f1:.4f}",
    "mAP@0.50": f"{map50:.4f}",
    "mAP@0.50:0.95": f"{map5095:.4f}",
})

df = pd.DataFrame(rows)
print("\n" + "=" * 82)
print("  HASIL EVALUASI TEST SET (529 CITRA) - YOLO26n + CA (CLASSIFICATION-OPTIMIZED)")
print("=" * 82)
display(df)


### 6. Visualisasi Kurva Konvergensi HD & Confusion Matrix

In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display, Markdown

csv_path = os.path.join(PROJECT_RUNS, RUN_NAME, "results.csv")
if os.path.exists(csv_path):
    epochs_list, box_l, cls_l, prec_l, rec_l, m50_l, m95_l = [], [], [], [], [], [], []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            clean = {k.strip(): v.strip() for k, v in r.items() if k}
            try:
                epochs_list.append(int(float(clean.get("epoch", len(epochs_list) + 1))))
                box_l.append(float(clean.get("train/box_loss", 0.0)))
                cls_l.append(float(clean.get("train/cls_loss", 0.0)))
                prec_l.append(float(clean.get("metrics/precision(B)", 0.0)))
                rec_l.append(float(clean.get("metrics/recall(B)", 0.0)))
                m50_l.append(float(clean.get("metrics/mAP50(B)", 0.0)))
                m95_l.append(float(clean.get("metrics/mAP50-95(B)", 0.0)))
            except ValueError:
                continue

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.suptitle(f"Kurva Konvergensi YOLO26n + CA ClsOpt ({len(epochs_list)} Epoch, 640x640)", fontsize=15, fontweight="bold")
    axes[0, 0].plot(epochs_list, box_l, color="#1f77b4", lw=2); axes[0, 0].set_title("Box Loss", fontweight="bold"); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(epochs_list, cls_l, color="#d62728", lw=2); axes[0, 1].set_title("Classification Loss", fontweight="bold"); axes[0, 1].grid(True, alpha=0.3)
    axes[0, 2].plot(epochs_list, prec_l, color="#2ca02c", lw=2, label="Precision"); axes[0, 2].plot(epochs_list, rec_l, color="#e377c2", lw=2, label="Recall"); axes[0, 2].legend(); axes[0, 2].set_title("Precision & Recall", fontweight="bold"); axes[0, 2].grid(True, alpha=0.3)
    axes[1, 0].plot(epochs_list, m50_l, color="#17becf", lw=2); axes[1, 0].set_title("mAP@0.50", fontweight="bold"); axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(epochs_list, m95_l, color="#bcbd22", lw=2); axes[1, 1].set_title("mAP@0.50:0.95", fontweight="bold"); axes[1, 1].grid(True, alpha=0.3)
    f1_l = [2 * p * r / (p + r + 1e-9) for p, r in zip(prec_l, rec_l)]
    axes[1, 2].plot(epochs_list, f1_l, color="#8c564b", lw=2); axes[1, 2].set_title("F1-Score", fontweight="bold"); axes[1, 2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

display(Markdown("#### Confusion Matrix & PR Curve pada Test Set:"))
for p_img in [
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval/confusion_matrix_normalized.png",
    f"{PROJECT_RUNS}/{RUN_NAME}/test_eval/BoxPR_curve.png"
]:
    if os.path.exists(p_img):
        display(IPyImage(p_img, width=700))


### 7. Visualisasi Hasil Testing pada Held-Out Test Set (15 Gambar Testing Lintas 7 Kelas: Ground Truth vs Prediksi)

In [ ]:
import os
import glob
import random
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Jumlah sampel gambar testing yang ditampilkan (minimal 10; 15 = grid 3x5 penuh mewakili ke-7 kelas)
NUM_TEST_SAMPLES = 15
CONF_TEST_VIS = 0.15

if "register_ca_everywhere" in globals():
    register_ca_everywhere()
if "PROJECT_RUNS" not in globals():
    PROJECT_RUNS = "/content/phase3_cls_opt_runs"
if "RUN_NAME" not in globals():
    RUN_NAME = "yolo26n_ca_cls_opt_50ep"

# Otomatis temukan bobot best.pt (baik habis training maupun langsung dari backup Google Drive)
w_candidates = []
if "BEST_WEIGHTS" in globals() and os.path.exists(BEST_WEIGHTS):
    w_candidates.append(BEST_WEIGHTS)
w_candidates += (
    glob.glob(f"{PROJECT_RUNS}/{RUN_NAME}/weights/best.pt") +
    glob.glob(f"{PROJECT_RUNS}/**/weights/best.pt", recursive=True) +
    glob.glob("/content/drive/MyDrive/SKRIPSI/phase3_cls_opt_backup/**/weights/best.pt", recursive=True) +
    glob.glob("/content/drive/MyDrive/**/yolo26n_ca_cls_opt*/**/best.pt", recursive=True)
)
if not w_candidates:
    raise FileNotFoundError("Bobot best.pt belum ditemukan! Jalankan Cell 4 atau pastikan backup tersedia di Google Drive.")

vis_weights = w_candidates[0]
print(f"[INFO] Memuat bobot model untuk visualisasi testing: {vis_weights}")
vis_model = YOLO(vis_weights)

test_img_dir = os.path.join(DATA_DIR, "images", "test")
test_lbl_dir = os.path.join(DATA_DIR, "labels", "test")
all_test_imgs = sorted([
    p for p in glob.glob(os.path.join(test_img_dir, "*.*"))
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Kelompokkan citra Test Set per kelas agar seluruh 7 kelas terwakili secara proporsional
by_class = {c: [] for c in range(len(CLASS_NAMES))}
for img_p in all_test_imgs:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(test_lbl_dir, stem + ".txt")
    if not os.path.exists(lbl_p):
        continue
    with open(lbl_p, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    if not lines:
        continue
    cid = int(lines[0].split()[0])
    gts = []
    for ln in lines:
        ps = ln.split()
        if len(ps) >= 5:
            gts.append((int(ps[0]), [float(x) for x in ps[1:5]]))
    if cid in by_class and gts:
        by_class[cid].append((img_p, gts))

rng = random.Random(42)
for cid in by_class:
    rng.shuffle(by_class[cid])

selected_samples = []
round_idx = 0
while len(selected_samples) < NUM_TEST_SAMPLES:
    added_in_round = False
    for cid in range(len(CLASS_NAMES)):
        if round_idx < len(by_class.get(cid, [])):
            selected_samples.append((cid, by_class[cid][round_idx][0], by_class[cid][round_idx][1]))
            added_in_round = True
            if len(selected_samples) >= NUM_TEST_SAMPLES:
                break
    if not added_in_round:
        break
    round_idx += 1

def calc_iou_xyxy(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    area2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    return inter / max(area1 + area2 - inter, 1e-6)

cols = 5
rows_grid = int(np.ceil(len(selected_samples) / float(cols)))
fig, axes = plt.subplots(rows_grid, cols, figsize=(22, 4.6 * rows_grid))
fig.suptitle(
    f"Hasil Pengujian YOLO26n + CA (ClsOpt) pada Held-Out Test Set ({len(selected_samples)} Gambar Lintas 7 Kelas)",
    fontsize=16,
    fontweight="bold",
)
axes_flat = np.array(axes).reshape(-1)

n_correct = 0
ious_vis, confs_vis = [], []

for idx, (gt_cid, img_p, gts) in enumerate(selected_samples):
    img_bgr = cv2.imread(img_p)
    H, W = img_bgr.shape[:2]
    vis_bgr = img_bgr.copy()

    # 1. Gambar kotak referensi Ground Truth (garis putih tipis)
    gt_boxes_px = []
    for g_c, (xc, yc, bw, bh) in gts:
        gx1, gy1 = int((xc - bw / 2.0) * W), int((yc - bh / 2.0) * H)
        gx2, gy2 = int((xc + bw / 2.0) * W), int((yc + bh / 2.0) * H)
        gt_boxes_px.append((g_c, [gx1, gy1, gx2, gy2]))
        cv2.rectangle(vis_bgr, (gx1, gy1), (gx2, gy2), (255, 255, 255), max(2, int(min(H, W) * 0.006)))

    # 2. Prediksi Model YOLO26n + CA pada citra Test Set
    res = vis_model.predict(img_bgr, conf=CONF_TEST_VIS, imgsz=640, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        res = vis_model.predict(img_bgr, conf=0.01, imgsz=640, verbose=False)[0]

    gt_name = CLASS_NAMES[gt_cid] if gt_cid < len(CLASS_NAMES) else f"Class {gt_cid}"
    if res.boxes is not None and len(res.boxes) > 0:
        confs_np = res.boxes.conf.cpu().numpy()
        clss_np = res.boxes.cls.cpu().numpy().astype(int)
        xyxy_np = res.boxes.xyxy.cpu().numpy()
        top_i = int(np.argmax(confs_np))
        p_cid = int(clss_np[top_i])
        p_conf = float(confs_np[top_i])
        px1, py1, px2, py2 = [int(v) for v in xyxy_np[top_i]]
        p_name = CLASS_NAMES[p_cid] if p_cid < len(CLASS_NAMES) else f"Class {p_cid}"
        col = CLASS_COLORS[p_cid % len(CLASS_COLORS)]
        txt_col = (15, 15, 15) if (0.114 * col[0] + 0.587 * col[1] + 0.299 * col[2]) > 160 else (255, 255, 255)

        thick = max(3, int(min(H, W) * 0.010))
        cv2.rectangle(vis_bgr, (px1, py1), (px2, py2), col, thick)
        badge = f"{p_name} {p_conf:.2f}"
        f_scale = max(0.50, min(H, W) / 900.0)
        (tw, th), _ = cv2.getTextSize(badge, cv2.FONT_HERSHEY_SIMPLEX, f_scale, 2)
        by_top = max(th + 8, py1)
        cv2.rectangle(vis_bgr, (px1, by_top - th - 8), (min(W - 1, px1 + tw + 8), by_top), col, -1)
        cv2.putText(vis_bgr, badge, (px1 + 4, by_top - 5), cv2.FONT_HERSHEY_SIMPLEX, f_scale, txt_col, 2, cv2.LINE_AA)

        best_iou = max([calc_iou_xyxy([px1, py1, px2, py2], gb) for _, gb in gt_boxes_px]) if gt_boxes_px else 0.0
        ious_vis.append(best_iou)
        confs_vis.append(p_conf)
        is_ok = (p_cid == gt_cid)
        if is_ok:
            n_correct += 1
        status_tag = "BENAR" if is_ok else "SALAH"
        pred_line = f"Pred: {p_name} ({p_conf*100:.1f}%) | IoU: {best_iou:.2f} [{status_tag}]"
        title_col = "#1b7837" if is_ok else "#d62728"
    else:
        pred_line = "Pred: Tidak Terdeteksi [FN]"
        title_col = "#d62728"

    ax = axes_flat[idx]
    ax.imshow(cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(f"[{idx+1}] GT: {gt_name}\n{pred_line}", fontsize=9.5, fontweight="bold", color=title_col)
    ax.axis("off")

for j in range(len(selected_samples), len(axes_flat)):
    axes_flat[j].axis("off")

plt.tight_layout(rect=[0, 0.01, 1, 0.95])
out_dir_vis = os.path.join(PROJECT_RUNS, RUN_NAME)
os.makedirs(out_dir_vis, exist_ok=True)
out_grid_path = os.path.join(out_dir_vis, f"{len(selected_samples)}_test_predictions_grid.png")
plt.savefig(out_grid_path, dpi=200, bbox_inches="tight")
plt.show()

mean_iou_v = float(np.mean(ious_vis)) if ious_vis else 0.0
mean_conf_v = float(np.mean(confs_vis)) if confs_vis else 0.0
print(f"[RINGKASAN TESTING VISUAL] Klasifikasi Benar: {n_correct}/{len(selected_samples)} ({100.0*n_correct/max(1,len(selected_samples)):.1f}%) | Rata-rata Conf: {mean_conf_v*100:.1f}% | Rata-rata IoU: {mean_iou_v:.3f}")
print(f"[OK] Gambar grid hasil testing disimpan di: {out_grid_path}")

if os.path.exists("/content/drive/MyDrive"):
    backup_dest = f"/content/drive/MyDrive/SKRIPSI/phase3_cls_opt_backup/{RUN_NAME}"
    os.makedirs(backup_dest, exist_ok=True)
    shutil.copy2(out_grid_path, os.path.join(backup_dest, os.path.basename(out_grid_path)))
    print(f"[OK] Otomatis dicadangkan ke Google Drive: {backup_dest}/{os.path.basename(out_grid_path)}")


### 8. Engine Deteksi Lapangan dengan Strategi 3 (Pure Model-Driven 4-View TTA + Shadow CLAHE)

In [ ]:
import cv2

class TeaLeafFieldDetector:
    def __init__(self, weights_path: str):
        self.model = YOLO(weights_path)
        self.class_names = CLASS_NAMES
        self.class_colors = CLASS_COLORS

    def _get_leaf_mask(self, img_bgr):
        H, W = img_bgr.shape[:2]
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
        b, g, r = img_bgr[:, :, 0].astype(np.float32), img_bgr[:, :, 1].astype(np.float32), img_bgr[:, :, 2].astype(np.float32)
        exg = 2.0 * g - r - b

        is_woody = (h < 13) | (h > 150) | (r > g + 20.0) | ((r > g + 4.0) & (g - b < 32.0)) | (exg < 8.0)
        is_gray_twig = (s < 35) & (exg < 22.0)
        is_shadow = (v < 48)
        is_glossy_leaf = (h >= 18) & (h <= 88) & (s >= 20) & (v >= 85) & (g > r + 10.0) & (exg >= 14.0)

        is_leaf = (((h >= 13) & (h <= 90) & (s >= 35) & (v >= 48) & (exg >= 10.0) & (~is_gray_twig)) | is_glossy_leaf) & (~is_woody) & (~is_shadow)
        mask = (is_leaf.astype(np.uint8)) * 255
        k_close = max(7, int(min(H, W) * 0.013) | 1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close)))
        k_open = max(5, int(min(H, W) * 0.012) | 1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open)))
        return mask

    def _get_white_twig_mask(self, crop_bgr):
        hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
        s, v = hsv[:, :, 1].astype(np.float32), hsv[:, :, 2].astype(np.float32)
        b, g, r = crop_bgr[:, :, 0].astype(np.float32), crop_bgr[:, :, 1].astype(np.float32), crop_bgr[:, :, 2].astype(np.float32)
        exg = 2.0 * g - r - b
        is_white_twig = ((s < 62.0) & (v >= 82.0) & (np.abs(r - b) < 32.0) & (exg < 30.0)) | ((v >= 165.0) & (s < 88.0) & (r >= g - 12.0))
        m = (is_white_twig.astype(np.uint8)) * 255
        return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))

    def _normalize_leaf_lighting(self, crop_bgr, mask_u8):
        """STRATEGI 3a: Koreksi bayangan lembut (LAB-CLAHE) hanya jika daun gelap terkena bayangan semak."""
        if np.count_nonzero(mask_u8) < 30:
            return crop_bgr
        lab = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2LAB)
        l_ch, a_ch, b_ch = cv2.split(lab)
        if float(np.mean(l_ch[mask_u8 > 0])) < 112.0:
            clahe = cv2.createCLAHE(clipLimit=1.6, tileGridSize=(4, 4))
            l_new = cv2.addWeighted(clahe.apply(l_ch), 0.55, l_ch, 0.45, 0)
            return cv2.cvtColor(cv2.merge([l_new, a_ch, b_ch]), cv2.COLOR_LAB2BGR)
        return crop_bgr

    def segment_leaves(self, img_bgr):
        H, W = img_bgr.shape[:2]
        img_area = float(H * W)
        base_mask = self._get_leaf_mask(img_bgr)

        bg_blur = cv2.medianBlur(img_bgr, max(15, int(min(H, W) * 0.032) | 1))
        step_img = cv2.bilateralFilter(bg_blur, 7, 35, 35)

        pad = max(12, int(min(H, W) * 0.025))
        padded = cv2.copyMakeBorder(base_mask, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
        dist = cv2.distanceTransform(padded, cv2.DIST_L2, 5)[pad:pad+H, pad:pad+W]
        dist_s = cv2.GaussianBlur(dist, (15, 15), 3.0)

        nms_rad = max(7, int(min(H, W) * 0.035))
        k_nms = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (nms_rad * 2 + 1, nms_rad * 2 + 1))
        local_max = (dist_s == cv2.dilate(dist_s, k_nms)) & (dist_s >= 6.5) & (base_mask > 0)

        peak_ys, peak_xs = np.where(local_max)
        peaks = sorted([(float(dist[py, px]), int(px), int(py)) for py, px in zip(peak_ys, peak_xs)], reverse=True)

        markers = np.zeros((H, W), dtype=np.int32)
        markers[base_mask == 0] = 1
        seeds = []
        for idx_s, (r_peak, px, py) in enumerate(peaks, start=2):
            if any(np.hypot(px - sx, py - sy) < max(10.0, 0.62 * min(r_peak, sr)) for _, sr, sx, sy in seeds):
                continue
            seeds.append((idx_s, r_peak, px, py))
            cv2.circle(markers, (px, py), max(4, int(0.30 * r_peak)), idx_s, -1)

        cv2.watershed(step_img, markers)
        candidates = []
        yy, xx = np.ogrid[:H, :W]

        for inst_id, r_peak, px, py in seeds:
            m_inst = ((markers == inst_id) & (base_mask > 0)).astype(np.uint8) * 255
            if np.count_nonzero(m_inst) < int(img_area * 0.0065):
                continue
            rad_lim = max(32.0, 2.35 * r_peak)
            reach = (((xx - px) / rad_lim) ** 2 + ((yy - py) / rad_lim) ** 2) <= 1.0
            m_inst[~reach] = 0

            cnts, _ = cv2.findContours(m_inst, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts: continue
            c = max(cnts, key=cv2.contourArea)
            area = float(cv2.contourArea(c))
            perim = float(cv2.arcLength(c, True))
            circularity = (4.0 * np.pi * area) / (perim * perim + 1e-6)
            hull = cv2.convexHull(c)
            solidity = area / (float(cv2.contourArea(hull)) + 1e-6)
            bx, by, bw, bh = cv2.boundingRect(c)
            ar = max(bw, bh) / max(1.0, min(bw, bh))
            extent = area / float(max(1, bw * bh))
            box_green_ratio = float(np.mean(base_mask[by:by+bh, bx:bx+bw] > 0))

            if (area / img_area < 0.0065) or (area / img_area > 0.16) or (ar > 3.6) or (solidity < 0.68) or (circularity < 0.22) or (extent < 0.46) or (box_green_ratio < 0.45):
                continue

            crop = img_bgr[by:by+bh, bx:bx+bw]
            c_local = c.copy(); c_local[:, :, 0] -= bx; c_local[:, :, 1] -= by
            mask_c = np.zeros((bh, bw), dtype=np.uint8)
            cv2.drawContours(mask_c, [c_local], -1, 255, -1)
            inside = mask_c > 0

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).astype(np.float32)
            gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
            mag = np.hypot(gx, gy) / 4.0
            k_rng = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
            l_rng = cv2.dilate(gray, k_rng) - cv2.erode(gray, k_rng)

            ezones = inside & (l_rng >= 18.0)
            acutance = float(np.percentile(mag[ezones] / (l_rng[ezones] + 8.0), 80)) if np.count_nonzero(ezones) >= 20 else 0.25
            if acutance < 0.19:
                continue

            candidates.append((area * solidity, c, (bx, by, bx + bw, by + bh), solidity, m_inst))

        candidates.sort(key=lambda t: t[0], reverse=True)
        kept, kept_masks, kept_boxes = [], [], []
        for _, c, bbox, sol, m_cand in candidates:
            cand_px = float(np.count_nonzero(m_cand))
            bx1, by1, bx2, by2 = bbox
            b_area = float((bx2 - bx1) * (by2 - by1))
            overlap = False
            for m_prev, (px1, py1, px2, py2) in zip(kept_masks, kept_boxes):
                inter = float(np.count_nonzero((m_cand > 0) & (m_prev > 0)))
                if inter / max(cand_px, 1.0) > 0.22:
                    overlap = True; break
                ix1, iy1, ix2, iy2 = max(bx1, px1), max(by1, py1), min(bx2, px2), min(by2, py2)
                if ix2 > ix1 and iy2 > iy1:
                    if ((ix2 - ix1) * (iy2 - iy1)) / min(b_area, float((px2 - px1) * (py2 - py1))) > 0.45:
                        overlap = True; break
            if not overlap:
                kept.append((c, bbox, sol))
                kept_masks.append(m_cand)
                kept_boxes.append(bbox)
        return kept

    def classify_leaf_tta(self, img_bgr, cnt, bbox):
        """STRATEGI 3b: Klasifikasi Murni Berbasis Model dengan 4-View TTA (Tanpa Blokir Warna Kaku)."""
        H, W = img_bgr.shape[:2]
        x1, y1, x2, y2 = bbox
        pad = max(6, int(max(x2 - x1, y2 - y1) * 0.04))
        cx1, cy1, cx2, cy2 = max(0, x1 - pad), max(0, y1 - pad), min(W, x2 + pad), min(H, y2 + pad)
        crop = img_bgr[cy1:cy2, cx1:cx2].copy()
        ch, cw = crop.shape[:2]

        m_crop = np.zeros((ch, cw), dtype=np.uint8)
        cnt_shifted = cnt.copy(); cnt_shifted[:, :, 0] -= cx1; cnt_shifted[:, :, 1] -= cy1
        cv2.drawContours(m_crop, [cnt_shifted], -1, 255, -1)

        # White-twig healing + Koreksi pencahayaan bayangan
        w_twig = self._get_white_twig_mask(crop)
        clean_leaf = (m_crop > 0) & (w_twig == 0)
        med_bgr = np.median(crop[clean_leaf], axis=0).astype(np.uint8) if np.count_nonzero(clean_leaf) >= 40 else np.array([45, 145, 75], dtype=np.uint8)
        crop_healed = crop.copy()
        crop_healed[(m_crop > 0) & (w_twig > 0)] = med_bgr
        crop_healed = self._normalize_leaf_lighting(crop_healed, m_crop)

        paper_color = np.array([236, 238, 238], dtype=np.uint8)
        side = int(max(ch, cw) * 1.34)
        canvas = np.full((side, side, 3), paper_color, dtype=np.uint8)
        y_off, x_off = (side - ch) // 2, (side - cw) // 2
        iso = np.full_like(crop_healed, paper_color)
        iso[m_crop > 0] = crop_healed[m_crop > 0]
        canvas[y_off:y_off+ch, x_off:x_off+cw] = iso

        # 4-View TTA: Original, Flip-H, Flip-V, Rotate-90
        views = [
            canvas,
            cv2.flip(canvas, 1),
            cv2.flip(canvas, 0),
            cv2.rotate(canvas, cv2.ROTATE_90_CLOCKWISE),
        ]
        preds = self.model.predict(views, conf=0.05, imgsz=640, verbose=False)

        n_cls = len(self.class_names)
        scores = {c: [] for c in range(n_cls)}
        for p in preds:
            if p.boxes is not None and len(p.boxes) > 0:
                view_best = {}
                for c_id, c_conf in zip(p.boxes.cls.cpu().numpy().astype(int), p.boxes.conf.cpu().numpy()):
                    cid_i = int(c_id)
                    if 0 <= cid_i < n_cls:
                        view_best[cid_i] = max(view_best.get(cid_i, 0.0), float(c_conf))
                for cid_i, v_sc in view_best.items():
                    scores[cid_i].append(v_sc)

        # Konsensus lintas 4 orientasi TTA
        rank_scores = {}
        disp_scores = {}
        for cid_i, s_list in scores.items():
            if s_list:
                mean_s = float(np.mean(s_list))
                rank_scores[cid_i] = mean_s * (0.76 + 0.08 * len(s_list))
                disp_scores[cid_i] = min(0.99, mean_s * (1.0 + 0.02 * (len(s_list) - 1)))

        if rank_scores:
            best_c = max(rank_scores, key=lambda k: rank_scores[k])
            return (best_c, disp_scores[best_c], self.class_names[best_c])

        # Fallback murni ke model dengan threshold sangat rendah
        low_preds = self.model.predict([canvas], conf=0.005, imgsz=640, verbose=False)
        if low_preds and low_preds[0].boxes is not None and len(low_preds[0].boxes) > 0:
            best_c = int(low_preds[0].boxes.cls[0].item())
            best_s = float(low_preds[0].boxes.conf[0].item())
            return (best_c, best_s, self.class_names[best_c])
        return (6, 0.50, self.class_names[6])

    def blur_background(self, img_bgr, leaf_instances=None, ksize: int = 15, sigma: float = 2.15):
        if leaf_instances is None:
            leaf_instances = self.segment_leaves(img_bgr)
        H, W = img_bgr.shape[:2]
        fg_mask = np.zeros((H, W), dtype=np.uint8)
        for cnt, _, _ in leaf_instances:
            cv2.drawContours(fg_mask, [cnt], -1, 255, -1)
        fg_mask = cv2.dilate(fg_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        alpha = (cv2.GaussianBlur(fg_mask, (7, 7), 0).astype(np.float32) / 255.0)[:, :, None]
        k = ksize | 1
        bg_blur = cv2.GaussianBlur(img_bgr, (k, k), sigma)
        return (img_bgr * alpha + bg_blur * (1.0 - alpha)).astype(np.uint8)

    def detect_and_visualize(self, img_bgr):
        H, W = img_bgr.shape[:2]
        leaf_instances = self.segment_leaves(img_bgr)
        img_bokeh = self.blur_background(img_bgr, leaf_instances)
        img_bbox = img_bokeh.copy()
        img_contour = img_bokeh.copy()
        overlay = img_contour.copy()

        n_diseased, n_healthy = 0, 0
        for cnt, bbox, sol in leaf_instances:
            cid, conf, cname = self.classify_leaf_tta(img_bgr, cnt, bbox)
            if cid == 7: continue
            color = self.class_colors[cid % len(self.class_colors)]
            if cid == 6: n_healthy += 1
            else: n_diseased += 1

            x1, y1, x2, y2 = bbox
            cv2.rectangle(img_bbox, (x1, y1), (x2, y2), color, 3)
            lbl = f"{cname} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
            cv2.rectangle(img_bbox, (x1, max(th + 6, y1) - th - 6), (min(W - 1, x1 + tw + 6), max(th + 6, y1)), color, -1)
            cv2.putText(img_bbox, lbl, (x1 + 3, max(th + 6, y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)

            cv2.fillPoly(overlay, [cnt], color)
            cv2.drawContours(img_contour, [cnt], -1, color, 3)
            cv2.putText(img_contour, lbl, (max(4, int(cnt[:, 0, 0].min())), max(18, int(cnt[:, 0, 1].min()))), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)

        cv2.addWeighted(overlay, 0.24, img_contour, 0.76, 0, img_contour)

        clean_mask = self._get_leaf_mask(img_bgr)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        canny_leaf = cv2.bitwise_and(cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 30, 100), clean_mask)

        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title("1. Gambar Asli (Input Eksternal)", fontsize=13, fontweight="bold"); axes[0, 0].axis("off")
        axes[0, 1].imshow(cv2.cvtColor(img_bbox, cv2.COLOR_BGR2RGB)); axes[0, 1].set_title(f"2. YOLO26n+CA 4-View TTA ({len(leaf_instances)} Daun: {n_diseased} Sakit, {n_healthy} Sehat)", fontsize=13, fontweight="bold"); axes[0, 1].axis("off")
        axes[1, 0].imshow(cv2.cvtColor(img_contour, cv2.COLOR_BGR2RGB)); axes[1, 0].set_title("3. Edge Contour Daun Fokus (Bokeh)", fontsize=13, fontweight="bold"); axes[1, 0].axis("off")
        axes[1, 1].imshow(canny_leaf, cmap="gray"); axes[1, 1].set_title("4. Canny Edge Map (Ranting & Blur Dieliminasi)", fontsize=13, fontweight="bold"); axes[1, 1].axis("off")
        plt.tight_layout()
        plt.show()

detector = TeaLeafFieldDetector(BEST_WEIGHTS)
print("[OK] TeaLeafFieldDetector (4-View Pure Neural TTA + Shadow CLAHE) siap digunakan!")


### 9. Uji Coba Interaktif Gambar Kebun Sendiri (Upload Gambar)

In [ ]:
from google.colab import files

detector = TeaLeafFieldDetector(BEST_WEIGHTS)
print("Silakan pilih gambar daun teh dari perangkat Anda (.jpg / .png):")
uploaded = files.upload()

for fname in uploaded.keys():
    print(f"\nMemproses gambar: {fname} ...")
    img_bgr = cv2.imread(fname)
    if img_bgr is not None:
        if max(img_bgr.shape[:2]) > 1280:
            sc = 1280.0 / max(img_bgr.shape[:2])
            img_bgr = cv2.resize(img_bgr, (int(img_bgr.shape[1] * sc), int(img_bgr.shape[0] * sc)))
        detector.detect_and_visualize(img_bgr)
    else:
        print(f"[ERROR] Gagal membaca gambar: {fname}")


### 10. Backup Otomatis Hasil & Bobot Terbaik ke Google Drive

In [ ]:
BACKUP_DIR = "/content/drive/MyDrive/SKRIPSI/phase3_cls_opt_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

target_run = os.path.join(PROJECT_RUNS, RUN_NAME)
if os.path.exists(target_run):
    dest_dir = os.path.join(BACKUP_DIR, RUN_NAME)
    shutil.copytree(target_run, dest_dir, dirs_exist_ok=True)
    print(f"[SUKSES] Folder hasil {RUN_NAME} berhasil dicadangkan ke Google Drive:")
    print(f"         {dest_dir}")
    !ls -lh "{dest_dir}/weights"
else:
    print(f"[WARNING] Folder {target_run} belum ditemukan.")


### 11. Pra-Label Otomatis Data Lapangan dengan Model Klasifikasi Teroptimasi

In [ ]:
FIELD_DIR = "/content/drive/MyDrive/SKRIPSI/data_lapangan"
OUT_DIR   = "/content/roboflow_prelabeled_cls_opt"
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{OUT_DIR}/classes.txt", "w") as f:
    f.write("\n".join(CLASS_NAMES[:7] + ["Ranting"]) + "\n")

img_files = sorted(glob.glob(f"{FIELD_DIR}/*.*"))
for idx, img_path in enumerate(img_files, 1):
    img = cv2.imread(img_path)
    if img is None: continue
    if max(img.shape[:2]) > 1280:
        sc = 1280.0 / max(img.shape[:2])
        img = cv2.resize(img, (int(img.shape[1]*sc), int(img.shape[0]*sc)))
    H, W = img.shape[:2]
    leaves = detector.segment_leaves(img)
    img_blur = detector.blur_background(img, leaves)
    cv2.imwrite(f"{OUT_DIR}/lapangan_{idx:03d}.jpg", img_blur)

    lines = []
    for cnt, (x1, y1, x2, y2), _ in leaves:
        cls_id, _, _ = detector.classify_leaf_tta(img, cnt, (x1, y1, x2, y2))
        lines.append(f"{cls_id} {((x1+x2)/2)/W:.6f} {((y1+y2)/2)/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")

    with open(f"{OUT_DIR}/lapangan_{idx:03d}.txt", "w") as f:
        f.write("\n".join(lines))

zip_out = shutil.make_archive("/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_cls_opt", "zip", OUT_DIR)
print(f"[SUKSES] {len(img_files)} foto lapangan selesai di-prelabel -> {zip_out}")


### 12. Gabungkan Data Lapangan Berlabel & Fine-Tuning YOLO26n + CA

In [ ]:
RF_ZIP = "/content/drive/MyDrive/SKRIPSI/roboflow_lapangan.zip"
PRE_ZIP = "/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_cls_opt.zip"

if os.path.exists(RF_ZIP):
    shutil.unpack_archive(RF_ZIP, "/content/rf_data")
    for rf_sp, dst_sp in [("train", "train"), ("valid", "val"), ("test", "test")]:
        for kind in ["images", "labels"]:
            for p in glob.glob(f"/content/rf_data/{rf_sp}/{kind}/*.*"):
                shutil.copy2(p, f"{DATA_DIR}/{kind}/{dst_sp}/rf_{os.path.basename(p)}")
    CLASS_NAMES = CLASS_NAMES[:7] + ["Ranting"]
    with open(DATA_YAML, "w") as f:
        f.write(f"path: {DATA_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}\n")
    with open(MODEL_YAML, "r") as f: y_txt = f.read().replace("nc: 7", f"nc: {len(CLASS_NAMES)}")
    with open(MODEL_YAML, "w") as f: f.write(y_txt)
    print("[SUKSES] Data lapangan hasil koreksi Roboflow berhasil digabung!")
else:
    if not os.path.exists("/content/roboflow_prelabeled_cls_opt") and os.path.exists(PRE_ZIP):
        shutil.unpack_archive(PRE_ZIP, "/content/roboflow_prelabeled_cls_opt")
    imgs = sorted(glob.glob("/content/roboflow_prelabeled_cls_opt/*.jpg"))
    for i, img_p in enumerate(imgs):
        txt_p = img_p.replace(".jpg", ".txt")
        sp = "val" if i % 5 == 0 else "train"
        shutil.copy2(img_p, f"{DATA_DIR}/images/{sp}/{os.path.basename(img_p)}")
        if os.path.exists(txt_p): shutil.copy2(txt_p, f"{DATA_DIR}/labels/{sp}/{os.path.basename(txt_p)}")
    print(f"[SUKSES] {len(imgs)} foto lapangan berhasil dimasukkan ke dataset!")

for cache_f in glob.glob(f"{DATA_DIR}/labels/*.cache"):
    os.remove(cache_f)

s1_weights = (
    glob.glob(f"{PROJECT_RUNS}/yolo26n_ca_cls_opt_50ep/weights/best.pt") +
    glob.glob("/content/drive/MyDrive/SKRIPSI/**/yolo26n_ca_cls_opt_50ep/**/best.pt", recursive=True)
)
init_ft = s1_weights[0] if s1_weights else BEST_WEIGHTS
FT_RUN_NAME = "yolo26n_ca_cls_opt_finetune"
ft_model = YOLO(MODEL_YAML)
ft_model.train(
    data=DATA_YAML,
    pretrained=init_ft,
    epochs=30,
    imgsz=640,
    batch=16,
    workers=8,
    device="cuda" if torch.cuda.is_available() else "cpu",
    project=PROJECT_RUNS,
    name=FT_RUN_NAME,
    exist_ok=True,
    optimizer="AdamW",
    lr0=0.0004,
    lrf=0.01,
    cos_lr=True,
    box=8.0,
    cls=1.35,
    hsv_h=0.004,
    hsv_s=0.35,
    hsv_v=0.35,
    degrees=30.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.0,
    close_mosaic=30,
    freeze=4,
    seed=42,
    plots=True,
)
FT_BEST_WEIGHTS = os.path.join(PROJECT_RUNS, FT_RUN_NAME, "weights", "best.pt")
print(f"[SUKSES] Fine-Tuning selesai: {FT_BEST_WEIGHTS}")


### 13. Evaluasi Tuning (Metrik + Perbandingan Kiri Data Berlabel vs Kanan Prediksi Fine-Tuning)

In [ ]:
w_pt = FT_BEST_WEIGHTS if "FT_BEST_WEIGHTS" in globals() else BEST_WEIGHTS
ft_eval = YOLO(w_pt).val(data=DATA_YAML, split="val", imgsz=640, verbose=False)
mp, mr, m50, m95 = float(ft_eval.box.mp), float(ft_eval.box.mr), float(ft_eval.box.map50), float(ft_eval.box.map)
print(f"[HASIL FINE-TUNING YOLO26n+CA ClsOpt] Precision: {mp:.4f} | Recall: {mr:.4f} | F1: {2*mp*mr/(mp+mr+1e-9):.4f} | mAP@0.50: {m50:.4f} | mAP@0.50:0.95: {m95:.4f}")

PRE_DIR = "/content/roboflow_prelabeled_cls_opt"
if not os.path.exists(PRE_DIR) and os.path.exists("/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_cls_opt.zip"):
    shutil.unpack_archive("/content/drive/MyDrive/SKRIPSI/roboflow_prelabeled_lapangan_cls_opt.zip", PRE_DIR)

detector_ft = TeaLeafFieldDetector(w_pt)
sample_imgs = sorted(glob.glob(f"{PRE_DIR}/*.jpg"))[:10]

for img_p in sample_imgs:
    img = cv2.imread(img_p)
    if img is None: continue
    H, W = img.shape[:2]
    img_gt = img.copy()
    txt_p = img_p.replace(".jpg", ".txt")

    n_gt = 0
    if os.path.exists(txt_p):
        for line in open(txt_p).read().splitlines():
            parts = line.split()
            if len(parts) != 5: continue
            cid = int(parts[0])
            xc, yc, bw, bh = [float(x) for x in parts[1:]]
            x1, y1, x2, y2 = int((xc - bw/2)*W), int((yc - bh/2)*H), int((xc + bw/2)*W), int((yc + bh/2)*H)
            col = CLASS_COLORS[cid % len(CLASS_COLORS)]
            cv2.rectangle(img_gt, (x1, y1), (x2, y2), col, 3)
            cv2.putText(img_gt, CLASS_NAMES[cid], (x1, max(20, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
            n_gt += 1

    leaves = detector_ft.segment_leaves(img)
    img_pred = detector_ft.blur_background(img, leaves)
    n_pred = 0
    for cnt, (x1, y1, x2, y2), _ in leaves:
        cid, conf, cname = detector_ft.classify_leaf_tta(img, cnt, (x1, y1, x2, y2))
        if cid == 7: continue
        col = CLASS_COLORS[cid % len(CLASS_COLORS)]
        cv2.rectangle(img_pred, (x1, y1), (x2, y2), col, 3)
        cv2.putText(img_pred, f"{cname} {conf:.2f}", (x1, max(20, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        n_pred += 1

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"Data Berlabel: {os.path.basename(img_p)} ({n_gt} Daun)", fontweight="bold"); axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"Hasil Prediksi YOLO26n+CA ClsOpt ({n_pred} Daun)", fontweight="bold"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
